# 🎯 FloodShield AI — Model 5: Composite District Risk Score
**Output:** Low / Medium / High / Critical
**Logic:** Model 2 (Disease Outbreak) output + Flood Severity মিলিয়ে composite score

### কীভাবে চালাবে:
1. Drive-এ `disease_prediction_dataset_fixed.xlsx` থাকতে হবে
2. Drive-এ `FloodShieldAI/models/disease_outbreak_model.pkl` (Model 2 থেকে) থাকতে হবে
3. Runtime → Run all


## 1️⃣ Setup

In [ ]:
!pip install pandas openpyxl joblib scikit-learn -q
from google.colab import drive
drive.mount('/content/drive')


## 2️⃣ Load Dataset + Model 2 Artifacts

In [ ]:
import pandas as pd
import numpy as np
import joblib

BASE = '/content/drive/MyDrive/FloodShieldAI'

dp = pd.read_excel(f'{BASE}/data/disease_prediction_dataset_fixed.xlsx')

outbreak_model = joblib.load(f'{BASE}/models/disease_outbreak_model.pkl')
outbreak_le = joblib.load(f'{BASE}/models/outbreak_risk_label_encoder.pkl')
outbreak_features = joblib.load(f'{BASE}/models/outbreak_model_features.pkl')

print("Disease data:", dp.shape)
print("Outbreak model loaded:", type(outbreak_model))


## 3️⃣ Composite Risk Score Logic
- Disease Outbreak Risk (Model 2): Medium=1, High=2, Critical=3
- Flood Severity (flood_depth_cm থেকে, current observed condition — leakage না): None=0, Minor=1, Moderate=2, Severe=3
- **Composite = 0.6×outbreak_score + 0.4×flood_score**
- Final category: Low (<1) / Medium (1-2) / High (2-2.5) / Critical (2.5+)


In [ ]:
def flood_severity_score(depth_cm):
    if depth_cm < 10: return 0
    elif depth_cm < 40: return 1
    elif depth_cm < 80: return 2
    else: return 3

def compute_risk_score(row):
    X_row = row[outbreak_features].to_frame().T
    probs = outbreak_model.predict_proba(X_row)[0]
    outbreak_class = outbreak_le.classes_[probs.argmax()]
    outbreak_map = {'Medium':1, 'High':2, 'Critical':3}
    outbreak_score = outbreak_map.get(outbreak_class, 1)
    flood_score = flood_severity_score(row['flood_depth_cm'])
    composite = 0.6*outbreak_score + 0.4*flood_score

    if composite < 1: level = 'Low'
    elif composite < 2: level = 'Medium'
    elif composite < 2.5: level = 'High'
    else: level = 'Critical'

    return pd.Series({
        'predicted_outbreak_risk': outbreak_class,
        'flood_severity_score': flood_score,
        'composite_score': round(composite, 2),
        'final_risk_level': level
    })

sample = dp.sample(200, random_state=42).copy()
season_dummies = pd.get_dummies(sample['season'], prefix='season')
sample_full = pd.concat([sample, season_dummies], axis=1)
for col in outbreak_features:
    if col not in sample_full.columns:
        sample_full[col] = 0

risk_results = sample_full.apply(compute_risk_score, axis=1)
sample_out = pd.concat([sample[['district','date','season']], risk_results], axis=1)
print(sample_out['final_risk_level'].value_counts())
sample_out.head(10)


## 4️⃣ Reusable Function (single input — backend API-তে ব্যবহারের জন্য)

In [ ]:
def get_risk_score(rainfall_mm, temperature_c, humidity_percent, river_level_m, flood_depth_cm,
                    population_density, sanitation_score, water_quality_index,
                    previous_week_cases, previous_two_week_cases, season='Monsoon'):
    input_dict = {
        'rainfall_mm': rainfall_mm, 'temperature_c': temperature_c,
        'humidity_percent': humidity_percent, 'river_level_m': river_level_m,
        'flood_depth_cm': flood_depth_cm, 'population_density': population_density,
        'sanitation_score': sanitation_score, 'water_quality_index': water_quality_index,
        'previous_week_cases': previous_week_cases, 'previous_two_week_cases': previous_two_week_cases
    }
    season_cols = [c for c in outbreak_features if c.startswith('season_')]
    for col in season_cols:
        season_name = col.replace('season_', '')
        input_dict[col] = 1 if season_name == season else 0

    input_df = pd.DataFrame([input_dict])[outbreak_features]
    probs = outbreak_model.predict_proba(input_df)[0]
    outbreak_class = outbreak_le.classes_[probs.argmax()]
    outbreak_map = {'Medium':1, 'High':2, 'Critical':3}
    outbreak_score = outbreak_map.get(outbreak_class, 1)
    flood_score = flood_severity_score(flood_depth_cm)
    composite = 0.6*outbreak_score + 0.4*flood_score

    if composite < 1: level = 'Low'
    elif composite < 2: level = 'Medium'
    elif composite < 2.5: level = 'High'
    else: level = 'Critical'

    return {
        'predicted_outbreak_risk': outbreak_class,
        'flood_severity': ['None','Minor','Moderate','Severe'][flood_score],
        'composite_score': round(composite, 2),
        'final_risk_level': level
    }

print(get_risk_score(rainfall_mm=220, temperature_c=29, humidity_percent=87, river_level_m=7.5,
                      flood_depth_cm=55, population_density=2100, sanitation_score=0.35,
                      water_quality_index=50, previous_week_cases=210, previous_two_week_cases=190,
                      season='Monsoon'))


## 5️⃣ Save

In [ ]:
import os
SAVE_DIR = f'{BASE}/models'
os.makedirs(SAVE_DIR, exist_ok=True)
print("Model 5 uses Model 2's saved artifacts directly — no new file to save except this function's logic (already in this notebook).")
print("✅ Model 5 ready — get_risk_score() function can be copied directly into backend API code.")


---
## ✅ পরের ধাপ
`get_risk_score()` ফাংশন backend-এর `/predict/risk` endpoint-এ ব্যবহার হবে, এবং Model 6 (Recommendation)-এ input হিসেবে যাবে।

**সৎ Note:** এটা "trained ML model" না, domain-logic-driven composite scoring — explainable ও audit-করা যায় বলে সরকার/হাসপাতাল client-দের কাছে বেশি বিশ্বাসযোগ্য।
